<a href="https://colab.research.google.com/github/alexaK88/Q_jpeg_pennylane/blob/main/training_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pennylane
!pip install pennylane pennylane-lightning[gpu]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.3/913.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.5/366.5 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.7/39.7 

In [2]:
import pennylane as qml
from pennylane import numpy as np
from pennylane.templates import StronglyEntanglingLayers
import torch
from torch.nn.functional import relu
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from skimage.transform import resize


### Dataset Preparation


First, we load the MNIST dataset from openML.
- X is the pixel data
- y is the labels
- converting everything to `uint8` here to ensure all values are integers in [0, 255]

In [3]:
# loading mnist from keras.datasets
from keras.datasets import mnist
(X_train_full, y_train_full), (X_test_full, y_test_full) = mnist.load_data()
X = np.concatenate((X_train_full, X_test_full), axis=0)
y = np.concatenate((y_train_full, y_test_full), axis=0)

X = X.astype(np.uint8) # better to convert for binerization
y = y.astype(np.uint8)
# digits = load_digits()

# X = digits.data        # shape (1797, 64)
# y = digits.target      # labels 0–9


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
X.shape

(70000, 28, 28)

Next, we focus on 2 classes, i.e. binary classification.
Here, I've been experimenting with different classes, and I stopped on 4 vs 9, cause they have more subtle difference in pixels, they are similar looking.

In [5]:
# focus on binary classification
mask = (y == 0) | (y == 1)

X, y = X[mask], y[mask]
X.shape

(14780, 28, 28)

In [6]:
print("Unique labels:", np.unique(y))

Unique labels: [0 1]


In [7]:
y = np.where(y == 0, -1, 1)
print("Unique labels:", np.unique(y))

Unique labels: [-1  1]


- I take only the first `n_samples`.
- I convert X to a NumPy array, and shuffling the data randomly

In [8]:
n_samples = 360 # restricting to 6000 samples for now

X = X.values if hasattr(X, "values") else X # safer conversion

X = X[:n_samples]
y = y[:n_samples]

Now, I normalise pixel intensities.
- [0, 255] -> [0, 1]
- reshaping images back to 2D for resizing, i.e to 28x28 array with float values between 0 and 1.

In [9]:
X = X / 255.0
X = X.reshape(-1, 28, 28)

print(X.shape)
print("Pixel range:", X.min(), X.max())

(360, 28, 28)
Pixel range: 0.0 1.0


And now I reduce images to 8x8 + flattening to (, 64)
- resize -> flatten -> normalize

In [10]:
# convert each 28x28 binarised image to 8x8, then flatten to length 64
def to_8x8_vector(img_row):
    img_8x8 = resize(
        img_row,
        (8, 8),
        anti_aliasing=False,
        preserve_range=True,
        order=1 # controlling interpolation
    )
    img_8x8 = img_8x8.flatten().astype(float)
    # img_8x8 -= img_8x8.mean()
    norm = np.linalg.norm(img_8x8)
    if norm > 0:
        img_8x8 /= norm
    else:
        img_8x8[0] = 1.0
      # should be shape (64,)
    return img_8x8

# apply transformation to all images
X_8x8 = np.array([to_8x8_vector(x) for x in X], dtype=float)
X_8x8.shape

(360, 64)

In [11]:
# sanity check, make sure no NaNs exist and all vectors are normalised, i.e. norm is around 1
print("Any NaNs?", np.isnan(X_8x8).any())
print("Norm check:", np.min(np.linalg.norm(X_8x8, axis=1)), np.max(np.linalg.norm(X_8x8, axis=1)))

Any NaNs? False
Norm check: 0.9999999999999998 1.0000000000000002


I'm gonna do the splitting here, and carry both representations consistently.
- qek inputs: (64,) flattened and normalized vectors, for quantum kernel embedding
- qjpeg: 28x28 images

In [12]:
idx = np.arange(n_samples)

idx_train, idx_test, y_train, y_test = train_test_split(
    idx, y, test_size=0.2, random_state=42, stratify=y, shuffle=True
)

# QEK inputs (8x8 -> 64 -> normed)
X_train_qek = X_8x8[idx_train]
X_test_qek  = X_8x8[idx_test]

print("QEK train/test:", X_train_qek.shape, X_test_qek.shape)
print("Labels train/test:", y_train.shape, y_test.shape)

QEK train/test: (288, 64) (72, 64)
Labels train/test: (288,) (72,)


Data preparation is done.

### Device

In [13]:
n_qubits = 6
dev_var = qml.device("lightning.qubit", wires=n_qubits)

### Quantum Model

In [14]:
@qml.qnode(dev_var, diff_method="parameter-shift")
def quantum_model(x, params):
    """A variational quantum model."""

    # embedding
    qml.AmplitudeEmbedding(
        x,
        wires=range(n_qubits),
        normalize=False  # already normalized
    )


    # trainable measurement
    StronglyEntanglingLayers(params, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))

def quantum_model_plus_bias(x, params, bias):
    """Adding a bias."""
    return quantum_model(x, params) + bias

def hinge_loss(predictions, targets):
    """Implements the hinge loss."""
    all_ones = torch.ones_like(targets)
    hinge_loss = all_ones - predictions * targets
    # trick: since the max(0,x) function is not differentiable,
    # use the mathematically equivalent relu instead
    hinge_loss = relu(hinge_loss)
    return hinge_loss

def quantum_model_train(n_layers, steps, batch_size):
    """Train the quantum model defined above."""

    params = np.random.random((n_layers, n_qubits, 3))
    params_torch = torch.tensor(params, requires_grad=True)
    bias_torch = torch.tensor(0.0, requires_grad=True)

    opt = torch.optim.Adam([params_torch, bias_torch], lr=0.1)

    loss_history = []
    for i in range(steps):

        batch_ids = np.random.choice(len(X_train_qek), batch_size)

        X_batch = X_train_qek[batch_ids]
        y_batch = y_train[batch_ids]

        X_batch_torch = torch.tensor(X_batch, requires_grad=False)
        y_batch_torch = torch.tensor(y_batch, requires_grad=False)

        def closure():
            opt.zero_grad()
            preds = torch.stack(
                [quantum_model_plus_bias(x, params_torch, bias_torch) for x in X_batch_torch]
            )
            loss = torch.mean(hinge_loss(preds, y_batch_torch))

            # bookkeeping
            current_loss = loss.detach().numpy().item()
            loss_history.append(current_loss)
            if i % 10 == 0:
                print("step", i, ", loss", current_loss)

            loss.backward()
            return loss

        opt.step(closure)

    return params_torch, bias_torch, loss_history


def quantum_model_predict(X_pred, trained_params, trained_bias):
    """Predict using the quantum model defined above."""

    p = []
    for x in X_pred:

        x_torch = torch.tensor(x)
        pred_torch = quantum_model_plus_bias(x_torch, trained_params, trained_bias)
        pred = pred_torch.detach().numpy().item()
        if pred > 0:
            pred = 1
        else:
            pred = -1

        p.append(pred)
    return p


In [15]:
params_1, bias_1, history_1 = quantum_model_train(
    n_layers=1,
    steps=150,
    batch_size=100
)

preds_1 = quantum_model_predict(X_test_qek, params_1, bias_1)
acc_1 = np.mean(preds_1 == y_test)

print("Validation accuracy (1 layer):", acc_1)

step 0 , loss 1.0531460046768188
step 10 , loss 0.8180185556411743
step 20 , loss 0.7965799570083618
step 30 , loss 0.6635019779205322
step 40 , loss 0.6168878078460693
step 50 , loss 0.7195267677307129
step 60 , loss 0.7627321481704712
step 70 , loss 0.7037903070449829
step 80 , loss 0.6710310578346252
step 90 , loss 0.7492455244064331
step 100 , loss 0.7623378038406372
step 110 , loss 0.7485914826393127
step 120 , loss 0.6389034390449524
step 130 , loss 0.6390812397003174
step 140 , loss 0.7584167718887329
Validation accuracy (1 layer): 0.5416666666666666


In [16]:
params_2, bias_2, history_2 = quantum_model_train(
    n_layers=2,
    steps=150,
    batch_size=100
)

preds_2 = quantum_model_predict(X_test_qek, params_2, bias_2)
acc_2 = np.mean(preds_2 == y_test)

print("Validation accuracy (2 layers):", acc_2)


step 0 , loss 1.1100704669952393
step 10 , loss 0.8817489147186279
step 20 , loss 0.7653483748435974
step 30 , loss 0.6829240322113037
step 40 , loss 0.6602676510810852
step 50 , loss 0.7904196381568909
step 60 , loss 0.5891467928886414
step 70 , loss 0.6488004922866821
step 80 , loss 0.6749690175056458
step 90 , loss 0.6854002475738525
step 100 , loss 0.7083490490913391
step 110 , loss 0.7858890295028687
step 120 , loss 0.7254486680030823
step 130 , loss 0.6604962944984436
step 140 , loss 0.6402263641357422
Validation accuracy (2 layers): 0.5694444444444444


In [17]:
params_4, bias_4, history_4 = quantum_model_train(
    n_layers=4,
    steps=150,
    batch_size=100
)

preds_4 = quantum_model_predict(X_test_qek, params_4, bias_4)
acc_4 = np.mean(preds_4 == y_test)

print("Validation accuracy (4 layers):", acc_4)

step 0 , loss 0.9950451850891113
step 10 , loss 0.7128093242645264
step 20 , loss 0.545746386051178
step 30 , loss 0.4609419107437134
step 40 , loss 0.4235111474990845
step 50 , loss 0.4349912703037262
step 60 , loss 0.3781757354736328
step 70 , loss 0.4505089819431305
step 80 , loss 0.4380148649215698
step 90 , loss 0.3996277153491974
step 100 , loss 0.49085476994514465
step 110 , loss 0.4143940806388855
step 120 , loss 0.44805803894996643
step 130 , loss 0.45032185316085815
step 140 , loss 0.4366471767425537
Validation accuracy (4 layers): 0.9583333333333334


In [18]:
params_5, bias_5, history_5 = quantum_model_train(
    n_layers=5,
    steps=150,
    batch_size=100
)

preds_5 = quantum_model_predict(X_test_qek, params_5, bias_5)
acc_5 = np.mean(preds_5 == y_test)

print("Validation accuracy (5 layers):", acc_5)

step 0 , loss 0.9523574113845825
step 10 , loss 0.5780074000358582
step 20 , loss 0.4680958092212677
step 30 , loss 0.43511831760406494
step 40 , loss 0.42930784821510315
step 50 , loss 0.40684211254119873
step 60 , loss 0.3676871061325073
step 70 , loss 0.3340267241001129
step 80 , loss 0.3538621962070465
step 90 , loss 0.34442904591560364
step 100 , loss 0.4106484353542328
step 110 , loss 0.3246234059333801
step 120 , loss 0.3952574133872986
step 130 , loss 0.38708144426345825
step 140 , loss 0.3457481861114502
Validation accuracy (5 layers): 0.9722222222222222
